In [1]:
import pandas as pd

In [2]:
df_old = pd.read_parquet("../dataset.parquet")
df_new = pd.read_parquet("../dataset_v1_fix_depth_update_leakage.parquet")

df = df_old.merge(df_new, on="timestamp", suffixes=("_old", "_new"))

In [8]:
df_old.shape, df.shape

((747336, 10), (747336, 19))

In [9]:
features = [
    "spread",
    "imbalance_1",
    "imbalance_5",
    "imbalance_10",
    "microprice_minus_mid",
    "delta_midprice",
]

for f in features:
    diff = df[f"{f}_new"] - df[f"{f}_old"]
    print(f, diff.abs().mean(), diff.abs().quantile(0.99))

spread 0.002732840382371517 0.0
imbalance_1 0.03009149861466955 0.5426661406676764
imbalance_5 0.029835375506632823 0.5311924830929298
imbalance_10 0.029458209679268356 0.513622220099198
microprice_minus_mid 0.0012346297704230447 0.0028667361959734084
delta_midprice 0.15162295139000398 5.488250000000117


In [10]:
label_diff = df["label_new"] - df["label_old"]

print("label mean abs diff:", label_diff.abs().mean())
print("label corr:", df["label_old"].corr(df["label_new"]))

label mean abs diff: 0.17806508718969782
label corr: 0.47878825891576465


In [14]:
signal_old = df["imbalance_1_old"]
signal_new = df["imbalance_1_new"]

label = df["label_new"]  # всегда используем causal label!

print("old signal corr:", signal_old.corr(label))
print("new signal corr:", signal_new.corr(label))

old signal corr: 0.14095270527439777
new signal corr: 0.13232718525307222


In [15]:
df["spread_diff"] = df["spread_new"] - df["spread_old"]

df["spread_diff"].value_counts().head(10)

spread_diff
 0.00    746256
-0.99        27
-1.84        22
-0.10        19
-0.01        17
-0.41        15
 0.10        15
 0.01        14
-0.91        14
-1.20        14
Name: count, dtype: int64